# Bước 4: Huấn luyện mô hình LightGBM Classifier

Notebook này thực hiện huấn luyện mô hình **LightGBM Classifier** để dự đoán xếp hạng học lực (A-F) của sinh viên dựa trên tập dữ liệu đã được tiền xử lý.


## 1. Import thư viện

In [ ]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from lightgbm import LGBMClassifier
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    learning_curve,
    train_test_split,
)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

print("Imported necessary libraries successfully!")


## 2. Đọc dữ liệu và chuẩn bị đặc trưng

In [26]:
# T?m ???ng d?n t?p processed data b?ng c?ch leo l?n c?c th? m?c cha
data_path = 'data/processed/Student_Performance_processed.csv'
for _ in range(5):
    if os.path.exists(data_path):
        break
    data_path = os.path.join('..', data_path)

df_encoded = pd.read_csv(data_path)

# Danh s?ch ??c tr?ng d?ng ?? d? ?o?n, bao g?m 3 c?t ?i?m m?n h?c
objective_features = [
    # Numerical features
    'age', 'study_hours', 'attendance_percentage',
    'math_score', 'science_score', 'english_score',
    # Binary features
    'school_type', 'internet_access', 'extra_activities',
    # Ordinal feature
    'parent_education',
    # One-Hot Encoded features
    'gender_male', 'gender_female', 'gender_other',
    'travel_time_<15 min', 'travel_time_15-30 min', 'travel_time_30-60 min', 'travel_time_>60 min',
    'study_method_notes', 'study_method_textbook', 'study_method_group study',
    'study_method_coaching', 'study_method_mixed', 'study_method_online videos'
]

# T?o Interaction Features t? h?nh vi h?c t?p
df_encoded['study_hours_x_attendance'] = df_encoded['study_hours'] * df_encoded['attendance_percentage']
df_encoded['study_hours_squared'] = df_encoded['study_hours'] ** 2
df_encoded['attendance_squared'] = df_encoded['attendance_percentage'] ** 2

# T?o ??c tr?ng t?ng h?p t? 3 c?t ?i?m m?n h?c
subject_score_features = ['math_score', 'science_score', 'english_score']
df_encoded['subject_score_mean'] = df_encoded[subject_score_features].mean(axis=1)
df_encoded['subject_score_min'] = df_encoded[subject_score_features].min(axis=1)
df_encoded['subject_score_max'] = df_encoded[subject_score_features].max(axis=1)
df_encoded['subject_score_range'] = df_encoded['subject_score_max'] - df_encoded['subject_score_min']
df_encoded['subject_score_std'] = df_encoded[subject_score_features].std(axis=1)

objective_features += [
    'study_hours_x_attendance',
    'study_hours_squared',
    'attendance_squared',
    'subject_score_mean',
    'subject_score_min',
    'subject_score_max',
    'subject_score_range',
    'subject_score_std',
]

print(f"Loaded processed data: {df_encoded.shape[0]} rows, {df_encoded.shape[1]} columns.")
print(f"Feature count: {len(objective_features)}")
print("Subject score features added:", subject_score_features)


Loaded processed data: 25000 rows, 29 columns.
Feature count: 23


## 3. Hu?n luy?n m? h?nh ph?n l?p (D? ?o?n nh?m h?c l?c A-F)

?? t?ng ?? ch?nh x?c, m? h?nh **LightGBM** s? d?ng c? nh?m ??c tr?ng h?c t?p/h?nh vi v? 3 c?t ?i?m m?n h?c: `math_score`, `science_score`, `english_score`. Notebook c?ng t?o th?m c?c ??c tr?ng t?ng h?p t? ?i?m nh? trung b?nh, th?p nh?t, cao nh?t v? ?? ch?nh l?ch gi?a c?c m?n.


In [27]:
# Nhãn mục tiêu phân lớp
y = df_encoded['final_grade']
X = df_encoded[objective_features]

print("Target grade distribution (final_grade):")
print(y.value_counts().sort_index())

# Phân chia Train/Test set (có stratify để giữ phân bố lớp đồng đều)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain set: {X_train.shape[0]} rows, Test set: {X_test.shape[0]} rows")


Target grade distribution (final_grade):
final_grade
a    1205
b    2696
c    6161
d    6311
e    5672
f    2955
Name: count, dtype: int64

Train set: 20000 rows, Test set: 5000 rows


In [ ]:
# Train LightGBM Classifier with a stronger multiclass configuration
model = LGBMClassifier(
    objective='multiclass',
    n_estimators=180,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=7,
    min_child_samples=50,
    subsample=0.9,
    subsample_freq=1,
    colsample_bytree=0.85,
    class_weight='balanced',
    reg_alpha=0.2,
    reg_lambda=5.0,
    random_state=42,
    n_jobs=1,
    verbose=-1
)
model.fit(X_train, y_train)
print("Training completed!")


## 4. Đánh giá mô hình

In [29]:
# ??nh gi? tr?n Test set
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
target_labels = sorted(y.unique())

print(f"Accuracy: {acc*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_labels))

# Ki?m tra overfitting b?ng Macro-F1 tr?n train/test
y_train_pred = model.predict(X_train)
f1_train = f1_score(y_train, y_train_pred, average='macro')
f1_test = f1_score(y_test, y_pred, average='macro')
print("\nOverfitting Check")
print(f"Macro-F1 Train : {f1_train:.4f}")
print(f"Macro-F1 Test  : {f1_test:.4f}")
print(f"Difference     : {f1_train - f1_test:.4f}")

# Cross-validation ?? ??nh gi? ?n ??nh h?n tr?n to?n b? d? li?u
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_f1_scores = cross_val_score(model, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
cv_acc_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
print("\nCross-Validation on full dataset (3-fold)")
print(f"CV F1 Scores: {[f'{score:.4f}' for score in cv_f1_scores]}")
print(f"Mean CV F1 Score: {cv_f1_scores.mean():.4f} (+/- {cv_f1_scores.std():.4f})")
print(f"CV Accuracy Scores: {[f'{score:.4f}' for score in cv_acc_scores]}")
print(f"Mean CV Accuracy: {cv_acc_scores.mean():.4f} (+/- {cv_acc_scores.std():.4f})")


Accuracy: 79.18%

Classification Report:
              precision    recall  f1-score   support

           a       0.68      0.87      0.76       241
           b       0.71      0.71      0.71       539
           c       0.82      0.79      0.81      1232
           d       0.81      0.78      0.80      1262
           e       0.80      0.79      0.80      1135
           f       0.80      0.86      0.83       591

    accuracy                           0.79      5000
   macro avg       0.77      0.80      0.78      5000
weighted avg       0.79      0.79      0.79      5000



In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=target_labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', 
            xticklabels=target_labels, yticklabels=target_labels)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - LightGBM Classifier')
plt.tight_layout()
plt.show()


In [ ]:
# Feature Importance
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

print("Top 15 most important features:")
for i in range(min(15, len(objective_features))):
    print(f"  {i+1}. {objective_features[indices[i]]} ({importances[indices[i]]:.4f})")

# V? bi?u ??
plt.figure(figsize=(12, 8))
top_n = min(15, len(objective_features))
top_indices = indices[:top_n]
plt.barh(range(top_n), importances[top_indices], align='center', color='#2ca25f')
plt.yticks(range(top_n), [objective_features[i] for i in top_indices])
plt.xlabel('Feature Importance')
plt.title('LightGBM - Top Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 5. Learning Curve

Vẽ đường học của LightGBM và lưu ảnh vào thư mục `results/`.


In [ ]:
# Learning Curve
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

train_sizes, train_scores, val_scores = learning_curve(
    estimator=model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring='f1_macro',
    train_sizes=np.linspace(0.2, 1.0, 5),
    n_jobs=1
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(train_sizes, train_mean, 'o-', label='Train F1')
plt.plot(train_sizes, val_mean, 's-', label='Validation F1')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2)
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2)
plt.xlabel('Number of Training Samples')
plt.ylabel('Macro F1')
plt.title('Learning Curve - LightGBM')
plt.grid(True)
plt.legend()
plt.tight_layout()

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
results_dir = os.path.join(project_root, 'results')
os.makedirs(results_dir, exist_ok=True)
learning_curve_path = os.path.join(results_dir, 'learning_curve_lightgbm.png')
plt.savefig(learning_curve_path, dpi=150)
plt.show()

print(f"Learning curve saved to: {learning_curve_path}")


## 6. Lưu mô hình


In [ ]:
# Lưu mô hình vào thư mục models/ ở gốc dự án
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
save_dir = os.path.join(project_root, 'models')
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, 'lightgbm_model.pkl')
joblib.dump(model, model_path)
print(f"LightGBM model saved successfully to: {model_path}!")
print(f"Model uses {len(objective_features)} features.")
